In [ ]:
import torch

print(f"PyTorch version: {torch.version}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA version: {torch.version.cuda}")
else:
    print("Running on CPU")


In [ ]:
# !pip install transformers torch torchvision pillow matplotlib

In [ ]:
# !pip install huggingface_hub
# from huggingface_hub import login
# login()


In [ ]:
# !git lfs install
# !git clone https://huggingface.co/datasets/chiragp26/RoadSocial

In [ ]:
# !pip install huggingface_hub
# from huggingface_hub import login
# login()
# HF_TOKEN=your_huggingface_token_here

for preprocessing the data

In [ ]:
import os
import json
import random
import re
from pathlib import Path

import cv2
from tqdm import tqdm

# Configuration for new archive dataset (image + mask pairs)
base_path = Path('./archive/IDD_RESIZED')
image_dir = base_path / 'image_archive'
mask_dir = base_path / 'mask_archive'
dataset_root = Path('./dataset')

# Keep same target subset size strategy as before
target_samples_per_split = {
    'train': 1005,
    'val': 350,
    'test': 145,
}

# Processing controls
save_mask_files = True
seed = 42

# Output structure
for split in target_samples_per_split:
    os.makedirs(dataset_root / split / 'metadata', exist_ok=True)
    os.makedirs(dataset_root / split / 'frames', exist_ok=True)
    if save_mask_files:
        os.makedirs(dataset_root / split / 'masks', exist_ok=True)

# Resume + logging files
processed_manifest_path = dataset_root / 'processed_images.json'
failed_manifest_path = dataset_root / 'failed_samples.json'

if processed_manifest_path.exists():
    with open(processed_manifest_path, 'r', encoding='utf-8') as f:
        processed_images = set(json.load(f))
else:
    processed_images = set()

if failed_manifest_path.exists():
    with open(failed_manifest_path, 'r', encoding='utf-8') as f:
        failed_samples = json.load(f)
else:
    failed_samples = []


def save_manifests():
    with open(processed_manifest_path, 'w', encoding='utf-8') as f:
        json.dump(sorted(list(processed_images)), f, indent=2)
    with open(failed_manifest_path, 'w', encoding='utf-8') as f:
        json.dump(failed_samples, f, indent=2)


def extract_index(name: str):
    m = re.search(r'(\d+)', name)
    return int(m.group(1)) if m else None


# Approximate urban scene labels for mask IDs (kept generic and robust)
MASK_ID_TO_NAME = {
    0: 'background',
    1: 'road',
    2: 'sidewalk',
    3: 'building',
    4: 'wall',
    5: 'fence',
    6: 'pole',
    7: 'traffic light',
    8: 'traffic sign',
    9: 'vegetation',
    10: 'terrain',
    11: 'sky',
    12: 'person',
    13: 'rider',
    14: 'car',
    15: 'truck',
    16: 'bus',
    17: 'train',
    18: 'motorcycle',
    19: 'bicycle',
}


def summarize_mask(mask_path: Path):
    mask = cv2.imread(str(mask_path), cv2.IMREAD_UNCHANGED)
    if mask is None:
        return []

    if len(mask.shape) == 3:
        mask = mask[:, :, 0]

    class_ids = sorted(int(v) for v in set(mask.flatten().tolist()))
    class_names = [MASK_ID_TO_NAME[c] for c in class_ids if c in MASK_ID_TO_NAME and c != 0]
    return class_names[:8]


def build_metadata(sample_stem: str, visible_classes):
    if visible_classes:
        visible_text = ', '.join(visible_classes)
        qa_question = f'What is visible in this road scene? {visible_text}.'
        description = f'Urban traffic scene containing {visible_text}.'
    else:
        qa_question = 'What is visible in this road scene? urban street environment.'
        description = 'Urban traffic street scene.'

    return {
        'VideoName': f'{sample_stem}.mp4',
        'DatasetSource': 'archive/IDD_RESIZED',
        'ImageID': sample_stem,
        'Description': description,
        'QAs': [
            {
                'question': qa_question,
                'answer': 'scene understanding',
            }
        ],
    }


if not image_dir.exists() or not mask_dir.exists():
    raise FileNotFoundError(
        f'Expected archive folders not found. Missing: {image_dir} or {mask_dir}'
    )

image_files = sorted(image_dir.glob('Image_*.png'))
mask_files = sorted(mask_dir.glob('Mask_*.png'))

image_by_idx = {}
for p in image_files:
    idx = extract_index(p.name)
    if idx is not None:
        image_by_idx[idx] = p

mask_by_idx = {}
for p in mask_files:
    idx = extract_index(p.name)
    if idx is not None:
        mask_by_idx[idx] = p

paired_indices = sorted(set(image_by_idx.keys()) & set(mask_by_idx.keys()))
if not paired_indices:
    raise RuntimeError('No matching image/mask pairs found in archive dataset.')

random.seed(seed)
random.shuffle(paired_indices)

requested_total = sum(target_samples_per_split.values())
selected_indices = paired_indices[: min(requested_total, len(paired_indices))]

if len(selected_indices) < requested_total:
    print(
        f'Warning: requested {requested_total} samples but only {len(selected_indices)} pairs available.'
    )

# Allocate splits with priority train -> val -> test using requested caps
allocations = {}
cursor = 0
for split, requested in target_samples_per_split.items():
    remaining = len(selected_indices) - cursor
    take = min(requested, max(remaining, 0))
    allocations[split] = selected_indices[cursor : cursor + take]
    cursor += take

for split, split_indices in allocations.items():
    print(
        f'\nProcessing split={split} | requested={target_samples_per_split[split]} | selected={len(split_indices)}'
    )

    processed_in_split = 0
    for idx in tqdm(split_indices, desc=f'{split} samples'):
        image_path = image_by_idx[idx]
        mask_path = mask_by_idx[idx]
        sample_stem = f'Image_{idx}'

        if sample_stem in processed_images:
            processed_in_split += 1
            continue

        try:
            image = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
            if image is None:
                failed_samples.append(
                    {
                        'split': split,
                        'sample': sample_stem,
                        'reason': f'Could not read image: {image_path.name}',
                    }
                )
                continue

            # Keep downstream format: <sample>_frame0.jpg
            frame_out = dataset_root / split / 'frames' / f'{sample_stem}_frame0.jpg'
            cv2.imwrite(str(frame_out), image)

            if save_mask_files:
                mask_out = dataset_root / split / 'masks' / f'{sample_stem}_mask.png'
                cv2.imwrite(str(mask_out), cv2.imread(str(mask_path), cv2.IMREAD_UNCHANGED))

            visible_classes = summarize_mask(mask_path)
            meta = build_metadata(sample_stem, visible_classes)
            meta_out = dataset_root / split / 'metadata' / f'{sample_stem}.json'
            with open(meta_out, 'w', encoding='utf-8') as f:
                json.dump(meta, f, indent=2)

            processed_images.add(sample_stem)
            processed_in_split += 1

            if processed_in_split % 100 == 0:
                save_manifests()

        except Exception as e:
            failed_samples.append(
                {
                    'split': split,
                    'sample': sample_stem,
                    'reason': str(e),
                }
            )

    save_manifests()
    print(f'Completed split {split}: processed={processed_in_split}')

print('\nPreprocessing complete.')
print(f'Processed samples tracked: {len(processed_images)}')
print(f'Failed samples tracked: {len(failed_samples)}')

Phase 1

In [ ]:
import torch
from PIL import Image
import os
import glob
from transformers import CLIPProcessor, CLIPModel

# STEP 1: LOAD CLIP MODEL
print("Loading CLIP model and processor...")
model_id = "openai/clip-vit-base-patch32"
model = CLIPModel.from_pretrained(model_id)
processor = CLIPProcessor.from_pretrained(model_id)
model.eval()  # Set to evaluation mode

# STEP 2: DEFINE DIRECT LABEL PROMPTS (no keyword dictionary needed)
labels = [
    "a street scene",
    "a road with vehicles",
    "a building",
    "a pedestrian on a sidewalk",
    "a traffic light",
    "a rainy road",
    "a highway"
]

# STEP 3: LOAD DATA FROM DATASET FOLDER
dataset_path = "./dataset/train/frames"
image_paths = sorted(glob.glob(os.path.join(dataset_path, "*.jpg")))[:5]  # First 5 for validation

if not image_paths:
    print(f"No images found in {dataset_path}. Please ensure preprocessing finished successfully.")
else:
    print(f"Found {len(image_paths)} images. Starting Phase 1 Inference...\n")

    for img_path in image_paths:
        image = Image.open(img_path).convert("RGB")

        # Direct CLIP image-text matching with base label prompts
        inputs = processor(text=labels, images=image, return_tensors="pt", padding=True)

        with torch.no_grad():
            outputs = model(**inputs)
            logits_per_image = outputs.logits_per_image[0]
            probs = logits_per_image.softmax(dim=0).cpu().numpy()

        best_idx = probs.argmax()
        best_label = labels[best_idx]
        confidence = probs[best_idx]

        print(f"Image: {os.path.basename(img_path)}")
        print(f"Predicted label: \"{best_label}\"")
        print(f"Confidence Score: {confidence:.4f}")
        print("Top Scores:")
        top_indices = probs.argsort()[::-1][:3]
        for i in top_indices:
            print(f" - {labels[i]}: {probs[i]:.4f}")
        print("-" * 40)

In [ ]:
Phase 2

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import os
import json
from transformers import CLIPProcessor, CLIPModel
from tqdm import tqdm

# 1. PREPARE DATASET CLASS
class RoadSocialDataset(Dataset):
    def __init__(self, dataset_root, split, processor):
        self.dataset_root = dataset_root
        self.split = split
        self.processor = processor
        self.data = []

        meta_dir = os.path.join(dataset_root, split, 'metadata')
        frame_dir = os.path.join(dataset_root, split, 'frames')

        # Map frames to their metadata (using the first frame for this phase)
        for meta_file in os.listdir(meta_dir):
            with open(os.path.join(meta_dir, meta_file), 'r') as f:
                meta = json.load(f)

            video_name = meta.get('VideoName', meta_file.replace('.json', '.mp4')).split('.')[0]
            # Finding the first frame extracted for this video
            frame_path = os.path.join(frame_dir, f"{video_name}_frame0.jpg")

            if os.path.exists(frame_path):
                # Use the first QA question as the text label for training alignment
                if meta.get('QAs') and len(meta['QAs']) > 0:
                    text = meta['QAs'][0].get('question', 'a street scene')
                    self.data.append((frame_path, text))

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        img_path, text = self.data[idx]
        image = Image.open(img_path).convert("RGB")
        inputs = self.processor(text=[text], images=image, return_tensors="pt", padding='max_length', truncation=True)
        return {
            'pixel_values': inputs['pixel_values'].squeeze(0),
            'input_ids': inputs['input_ids'].squeeze(0),
            'attention_mask': inputs['attention_mask'].squeeze(0)
        }

# 2. INITIALIZE MODEL & DATA
device = "cuda" if torch.cuda.is_available() else "cpu"
model_id = "openai/clip-vit-base-patch32"
model = CLIPModel.from_pretrained(model_id).to(device)
processor = CLIPProcessor.from_pretrained(model_id)

# Phase 2 Rule: Freeze encoders, train projection layers
for param in model.vision_model.parameters(): param.requires_grad = False
for param in model.text_model.parameters(): param.requires_grad = False

train_ds = RoadSocialDataset("./dataset", "train", processor)
train_loader = DataLoader(train_ds, batch_size=8, shuffle=True)

# 3. LOSS & OPTIMIZER
optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-5)
loss_img = nn.CrossEntropyLoss()
loss_txt = nn.CrossEntropyLoss()

# 4. TRAINING LOOP
print(f"Starting Phase 2 Training on {len(train_ds)} samples...")
epochs = 20
model.train()

for epoch in range(epochs):
    total_loss = 0
    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}"):
        optimizer.zero_grad()

        pixel_values = batch['pixel_values'].to(device)
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)

        outputs = model(input_ids=input_ids, pixel_values=pixel_values, attention_mask=attention_mask, return_loss=True)

        # Contrastive similarity matrix is internal to CLIPModel forward when return_loss=True
        loss = outputs.loss
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1} Average Loss: {total_loss/len(train_loader):.4f}")

print("Fine-tuning complete. Model is now optimized for the RoadSocial dataset.")

In [ ]:
Phase 3

In [ ]:
import torch
from PIL import Image
import os
import numpy as np
import matplotlib.pyplot as plt
import random
from glob import glob

# Helper for inference
def run_inference(image_path, labels):
    image = Image.open(image_path).convert("RGB")
    inputs = processor(text=labels, images=image, return_tensors="pt", padding=True).to(device)
    with torch.no_grad():
        outputs = model(**inputs)
        probs = outputs.logits_per_image.softmax(dim=1).cpu().numpy()[0]
    return image, probs

# --- PART 1: POST-TRAINING VISUALIZATION ---
print("--- Phase 3: Post-Training Inference Results ---")
val_labels = ["a street scene", "a road with vehicles", "a building", "a pedestrian on a sidewalk", "a traffic light", "a rainy road", "a highway"]
test_image_dir = "./dataset/train/frames"
test_images = sorted(glob(os.path.join(test_image_dir, "*.jpg")))[:2]

for img_path in test_images:
    img, probs = run_inference(img_path, val_labels)
    top_idx = np.argmax(probs)

    plt.figure(figsize=(10, 4))
    plt.subplot(1, 2, 1)
    plt.imshow(img)
    plt.title(f"Pred: {val_labels[top_idx]}\nConf: {probs[top_idx]:.2f}")
    plt.axis('off')

    plt.subplot(1, 2, 2)
    top3_idx = probs.argsort()[-3:][::-1]
    plt.barh([val_labels[i] for i in top3_idx], probs[top3_idx], color='skyblue')
    plt.title("Top 3 Scores")
    plt.show()

# --- PART 2: ZERO-SHOT GENERALIZATION ---
print("\n--- Phase 3: Zero-Shot Generalization Test ---")
zero_shot_labels = ["an urban intersection", "a car dashboard view", "nighttime city driving"]
unseen_images = sorted(glob("./dataset/test/frames/*.jpg"))[:2]

plt.figure(figsize=(12, 4))
for i, img_path in enumerate(unseen_images):
    img, probs = run_inference(img_path, zero_shot_labels)
    plt.subplot(1, 2, i+1)
    plt.imshow(img)
    plt.title(f"Unseen: {zero_shot_labels[np.argmax(probs)]}")
    plt.axis('off')
plt.show()

# --- PART 3: FINAL PIPELINE ACCURACY ---
print("\n--- Final Evaluation Summary (Sample Size: 50) ---")
refined_labels = val_labels + ["an unrelated or blurry image"]
CONFIDENCE_THRESHOLD, SAMPLE_SIZE = 0.35, 500
all_frames = glob("./dataset/*/frames/*.jpg")
sampled_frames = random.sample(all_frames, min(len(all_frames), SAMPLE_SIZE))

relevant_count, noise_count = 0, 0
for img_path in sampled_frames:
    _, probs = run_inference(img_path, refined_labels)
    top_idx = np.argmax(probs)
    if (probs[top_idx] < CONFIDENCE_THRESHOLD) or (refined_labels[top_idx] == "an unrelated or blurry image"):
        noise_count += 1
    else:
        relevant_count += 1

accuracy = (relevant_count / len(sampled_frames)) * 100
print(f"Total: {len(sampled_frames)} | Noise: {noise_count} | Relevant: {relevant_count} | Accuracy: {accuracy:.2f}%")

# Summary Chart
plt.figure(figsize=(6, 4))
plt.bar(['Relevant', 'Noise'], [relevant_count, noise_count], color=['green', 'red'])
plt.title(f"Final Dataset Distribution (Accuracy: {accuracy:.1f}%)")
plt.ylabel("Image Count")
plt.show()